# exp060_lgbm_capacity_pseudotail_public_features train

Train-side audit for LightGBM capacity pseudo-tail residual model with exp056 public feature families.


## Contents

1. Setup and configuration
2. Input artifact check
3. Public feature cross-fit audit
4. Metrics and artifacts


## 1. Setup and configuration


In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, load_config
from public_feature_model_audit import resolve_feature_path, run_audit, get_nested


In [ ]:
paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", config["experiment"]["name"])
print("Route:", config["experiment"]["route"])
print("Parent:", config["lineage"]["parent"])
print("Input artifact:", get_nested(config, "data.feature_path"))
print("Base estimator:", get_nested(config, "model.estimator"))
print("Variants:", [item["name"] for item in get_nested(config, "model.variants", [])])


## 2. Input artifact check


In [ ]:
feature_path = resolve_feature_path(paths, get_nested(config, "data.feature_path"))
print("Feature path:", feature_path)
print("Exists:", feature_path.exists())

preview = pd.read_csv(feature_path, nrows=5)
print("Rows preview:", len(preview))
print("Columns:", list(preview.columns))
if "pseudo_cutoff_fraction" in preview.columns:
    print("Preview cutoffs:", sorted(preview["pseudo_cutoff_fraction"].dropna().unique().tolist()))


## 3. Public feature cross-fit audit


In [ ]:
summary = run_audit(
    paths,
    config,
    feature_path,
    output_dir=paths.artifacts_dir,
    max_wells=None,
    max_train_rows_override=None,
    skip_exp026_control=True,
)
print(json.dumps(summary, indent=2))


## 4. Metrics and artifacts


In [ ]:
metrics = pd.read_csv(paths.artifacts_dir / "public_feature_metrics.csv")
buckets = pd.read_csv(paths.artifacts_dir / "public_feature_bucket_metrics.csv")
splits = pd.read_csv(paths.artifacts_dir / "public_feature_split_metrics.csv")
family_matrix = pd.read_csv(paths.artifacts_dir / "public_feature_family_matrix.csv")
feature_parity = pd.read_csv(paths.artifacts_dir / "public_feature_feature_parity_report.csv")

display(metrics.sort_values(["audit", "rmse"]).head(30))
display(family_matrix.sort_values(["audit", "rmse"]).head(30))
display(feature_parity.sort_values(["source", "feature"]).head(80))
display(buckets.sort_values(["audit", "candidate", "bucket"]).head(40))
display(splits.sort_values(["audit", "candidate", "split"]).head(40))
